# DIM_DATE

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
    DecimalType,
    TimestampType,
)

import datetime
import calendar

In [0]:
from pyspark import pipelines as dp

## Environment Variable

## Schema Definition

In [0]:
schema = StructType(
    [
        StructField(name="datum_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Datum als Nummer"}),
        StructField(name="datum_date", dataType=DateType(), nullable=False, metadata={"comment": "Datum als Date"}),
        StructField(
            name="wochentag_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Wochentag als Nummer"}
        ),
        StructField(
            name="wochentag_bez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Wochentag ausgeschrieben"},
        ),
        StructField(
            name="wochentag_kbez", dataType=StringType(), nullable=False, metadata={"comment": "Wochentag in Kurz"}
        ),
        StructField(
            name="tag_im_monat_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Tag im Monat"}
        ),
        StructField(
            name="tag_im_jahr_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Tag im Jahr"}
        ),
        StructField(
            name="tage_seit_1900", dataType=IntegerType(), nullable=False, metadata={"comment": "Tage seit 1900"}
        ),
        StructField(
            name="jahr_kw_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Jahr und KW Nummer"}
        ),
        StructField(name="kw_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Aktuelle KW Nummer"}),
        StructField(
            name="kw_bez", dataType=StringType(), nullable=False, metadata={"comment": "Kurzbezeichnung der KW"}
        ),
        StructField(
            name="jahr_monat_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Jahr und Monat als Nummer"},
        ),
        StructField(
            name="monat_bez", dataType=StringType(), nullable=False, metadata={"comment": "Monat ausgeschrieben"}
        ),
        StructField(
            name="monat_kbez", dataType=StringType(), nullable=False, metadata={"comment": "Monat Beschreibung in Kurz"}
        ),
        StructField(
            name="monatjahr_kbez", dataType=StringType(), nullable=False, metadata={"comment": "Monat und Jahr in Kurz"}
        ),
        StructField(name="monat_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Monat als Nummer"}),
        StructField(
            name="jahr_quartal_num",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Jahr und Quartal Nummer (YYYYQQ)"},
        ),
        StructField(
            name="quartal_bez",
            dataType=StringType(),
            nullable=False,
            metadata={"comment": "Quartal als Beschreibung mit Q1"},
        ),
        StructField(
            name="quartal_num", dataType=IntegerType(), nullable=False, metadata={"comment": "Quartal als Nummer"}
        ),
        StructField(
            name="jahr", dataType=IntegerType(), nullable=False, metadata={"comment": "Zeigt das Jahr des Datums an. "}
        ),
        StructField(
            name="islastofmonth",
            dataType=BooleanType(),
            nullable=False,
            metadata={"comment": "1 wenn es sich um den letzten Monat handelt"},
        ),
        StructField(
            name="time_to_end_of_month",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Anzahl Tage bis zum Ende des Monats"},
        ),
        StructField(
            name="time_to_end_of_year",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment": "Anzahl Tage bis zum Ende des Jahres"},
        ),
    ]
)

## Create List

In [0]:
start_date = datetime.date(2010,1,1)
end_date = datetime.date(2030,12,31)

date_list = []


def convert_tag_to_eu(current_date):
    if current_date.strftime("%w") == "0":
        return 7
    else:
        return int(current_date.strftime("%w"))


def translate_weekday(current_date):
    d = current_date.strftime("%A")
    if d == "Monday":
        return "Montag"
    elif d == "Tuesday":
        return "Dienstag"
    elif d == "Wednesday":
        return "Mittwoch"
    elif d == "Thursday":
        return "Donnerstag"
    elif d == "Friday":
        return "Freitag"
    elif d == "Saturday":
        return "Samstag"
    elif d == "Sunday":
        return "Sonntag"


def translate_weekday_short(current_date):
    d = current_date.strftime("%a")
    if d == "Mon":
        return "Mo"
    elif d == "Tue":
        return "Di"
    elif d == "Wed":
        return "Mi"
    elif d == "Thu":
        return "Do"
    elif d == "Fri":
        return "Fr"
    elif d == "Sat":
        return "Sa"
    elif d == "Sun":
        return "So"


# def last_da_of_month(current_date):
#   if current_date.day == calendar.monthrange(current_date.year, current_date.month)[1]:
#     return 1
#   else:
#     return 0


def translate_month(current_date):
    m = current_date.strftime("%B")
    if m == "January":
        return "Januar"
    elif m == "February":
        return "Februar"
    elif m == "March":
        return "März"
    elif m == "April":
        return "April"
    elif m == "May":
        return "Mai"
    elif m == "June":
        return "Juni"
    elif m == "July":
        return "Juli"
    elif m == "August":
        return "August"
    elif m == "September":
        return "September"
    elif m == "October":
        return "Oktober"
    elif m == "November":
        return "November"
    elif m == "December":
        return "Dezember"


def translate_month_kurz(current_date):
    m = current_date.strftime("%b")
    if m == "Mar":
        return "Mär"
    elif m == "May":
        return "Mai"
    elif m == "Oct":
        return "Okt"
    elif m == "Dec":
        return "Dez"
    else:
        return m


current_date = start_date
while current_date <= end_date:
    date_list.append(
        {
            "datum_num": int(current_date.strftime("%Y%m%d")),
            "datum_date": current_date,
            "wochentag_num": convert_tag_to_eu(current_date),
            "wochentag_bez": translate_weekday(current_date),
            "wochentag_kbez": translate_weekday_short(current_date),
            "tag_im_monat_num": current_date.day,
            "tag_im_jahr_num": int(current_date.strftime("%j")),
            "tage_seit_1900": (current_date - datetime.date(1900, 1, 1)).days,
            "jahr_kw_num": int(f"{current_date.isocalendar().year}{current_date.isocalendar().week:02}"),
            "kw_num": current_date.isocalendar().week,
            "kw_bez": f"KW {current_date.isocalendar().week:02}",
            "jahr_monat_num": int(f"{current_date.year}{current_date.strftime('%m')}"),
            "monat_bez": translate_month(current_date),
            "monat_kbez": translate_month_kurz(current_date),
            "monatjahr_kbez": f"{translate_month_kurz(current_date)} {current_date.strftime('%y')}",
            "monat_num": current_date.month,
            "jahr_quartal_num": int(f"{current_date.year}{(current_date.month - 1) // 3 + 1}"),
            "quartal_bez": f"Q {(current_date.month - 1) // 3 + 1}",
            "quartal_num": (current_date.month - 1) // 3 + 1,
            "jahr": current_date.year,
            #'islastofmonth' : last_da_of_month(current_date),
            "islastofmonth": current_date.day == calendar.monthrange(current_date.year, current_date.month)[1],
            "time_to_end_of_month": (calendar.monthrange(current_date.year, current_date.month)[1] - current_date.day)
            + 1,
            "time_to_end_of_year": 365 - int(current_date.strftime("%j")),
        }
    )
    current_date += datetime.timedelta(days=1)


## ETL/Create Date fields

In [0]:
@dp.temporary_view(name="dwh_datum_basis")
def bronze_dwh_datum_basis():
    df = spark.createDataFrame(data=date_list, schema=schema)
    return df


dp.create_streaming_table("analytics.bronze.dwh_datum", comment="This table shows the date table", schema=schema)

dp.create_auto_cdc_from_snapshot_flow(
    target="analytics.bronze.dwh_datum",
    source="dwh_datum_basis",
    keys=["datum_num"],
    stored_as_scd_type=1,
)